# Phase 3 · Two-stage document model (stage 1: sentence vectors from Phase 1, stage 2: BiLSTM/CRF over whole documents)

**Stage 1** encodes every sentence (cap 128 tokens, not 63) with the **frozen Phase 1 decoder**, once, and caches `[mean ‖ last-token]` vectors to Drive. Two pooling modes: `plain`, and `echo` (the sentence is fed twice and pooled from the second copy, so each token has seen the whole sentence despite the causal mask).
**Stage 2** trains `doclib.DocModel` (BiLSTM over the whole document + position + domain, optional CRF and label-shift auxiliary loss). Because stage 2 is cheap, it is evaluated with **grouped 5-fold CV over the 90 labelled documents** (pooled out-of-fold macro-F1), and the test prediction is the mean of the fold models' probabilities.

Needs `doclib.py` and `rr_lib.py` (v3) in `MyDrive/legal-llm-course/`. Cite: Kalamkar et al. 2022 (rhetorical roles, label-shift prediction), Springer et al. 2024 (echo embeddings), Lafferty et al. 2001 (CRF).

In [ ]:
# 1 · Settings
from pathlib import Path
ACTION   = 'Train'          # Inspect | Train | Predict   (Predict re-writes the submission from best.pt)
PRESET   = 'MLA-prenorm'    # GQA-prenorm | MHA-prenorm | MHA-postnorm | MLA-prenorm
METHOD   = 'lora'           # full | partial | lora | rslora | lorma | ia3 | prefix | kd
RUN_ID   = 'mla-lora-pos-001'
CONTEXT  = 'window'         # window | sentence
CLASS_WEIGHT = 'sqrt'       # none | sqrt | inverse
EPOCHS, SEED = 6, 17
USE_POS = True   # document-position features (new in rr_lib.py v2)

COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_3_RR'
RUNS_ROOT   = COURSE_ROOT/'runs/phase3_rr'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'   # existing Phase 2 bundle: legal_scaffold, training_methods, tokenizer
RR_LIB      = COURSE_ROOT/'rr_lib.py'                            # Phase 3 helper, upload next to the bundle

# adapter / method knobs
RANK, ALPHA = 8, 16                 # lora / rslora / lorma
LORA_TARGETS = ('query', 'value')   # mapped per architecture (MLA -> query_up/value_up)
PARTIAL_LAYERS = (10, 11)
PREFIX_LEN = 16
# knowledge distillation
KD_TEACHER_RUN = RUNS_ROOT/'MHA-prenorm/full/mha-full-win-001'   # finished `full` run used as teacher
KD_STUDENT_LAYERS = (0, 2, 4, 6, 8, 10)                            # 12 -> 6 blocks, copied from Phase 1
KD_TEMPERATURE, KD_ALPHA = 2.0, 0.5

DEFAULT_LR = {  # backbone / adapter / head learning rates
 'full': (5e-5, 0, 1e-3), 'kd': (5e-5, 0, 1e-3), 'partial': (1e-4, 0, 1e-3),
 'lora': (0, 1e-3, 1e-3), 'rslora': (0, 1e-3, 1e-3), 'lorma': (0, 1e-3, 1e-3),
 'ia3': (0, 3e-3, 1e-3), 'prefix': (0, 3e-3, 1e-3)}
assert ACTION in ('Inspect','Train','Predict') and METHOD in DEFAULT_LR and CONTEXT in ('window','sentence')

PRESETS = {
 'GQA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='gqa',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-postnorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='post',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MLA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='mla',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02,mla_latent=256,mla_query_rank=256,mla_rope_dim=32)}
# NOTE: presets copied from the Phase 2 notebook (same Phase 1 snapshots, same tokenizer).

# ---- two-stage settings ------------------------------------------------------------
STAGE1_PRESETS = ['MLA-prenorm', 'GQA-prenorm', 'MHA-prenorm']   # Phase 1 models to extract from
POOLINGS = ['plain', 'echo']
SENT_CAP = 128
VEC_DIR = COURSE_ROOT/'runs/phase3_rr/sentence_vectors'
STAGE2 = dict(preset='MLA-prenorm', pooling='echo')              # which cached vectors stage 2 uses
CV_EPOCHS, CV_K, CV_SEEDS = 60, 5, (0,)

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(COURSE_ROOT/'doclib.py', LOCAL/'doclib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data (downloads Phase_3_RR from the gated HF dataset if not already on Drive)
from google.colab import userdata
try: token = userdata.get('HF_TOKEN')
except Exception: token = None
R.ensure_files(COURSE_ROOT, token)
LABELS = R.load_labels(PHASE_DIR/'Train/label_vocab.json')
L2I = {l:i for i,l in enumerate(LABELS)}
DATA = R.load_all(PHASE_DIR, LABELS)
for s in DATA.values(): R.tokenize_docs(s, TOK, 64)
print(len(LABELS), 'labels:', LABELS)
print(json.dumps(R.summarize(DATA, LABELS), indent=1))
ids = [d['id'] for d in DATA['test']]
print('test ids unique across CL+IT:', len(ids)==len(set(ids)), '| test sentences:', sum(len(d['text']) for d in DATA['test']))
print('example:', DATA['train'][0]['text'][:2], DATA['train'][0]['labels'][:2])

In [ ]:
PHASE1_OVERRIDE = {'MHA-prenorm': str(COURSE_ROOT/'runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt')}   # same snapshot Phase 2 used; mha_pre is a Drive shortcut so auto-search misses it

def _scan_snapshots(preset):
    """Diagnostic: list every snapshot.json under runs/ (follows Drive shortcuts) vs the wanted config."""
    import os
    from legal_scaffold.models import ModelConfig
    want = ModelConfig(**PRESETS[preset]).validate().__dict__
    rows = []
    for dp, dn, fn in os.walk(COURSE_ROOT/'runs', followlinks=True):
        if 'snapshot.json' in fn:
            try:
                m = json.loads((Path(dp)/'snapshot.json').read_text())
                ident = m.get('identity', {}); cfg = ident.get('model', ident.get('config')) or {}
                got = ModelConfig(**cfg).validate().__dict__ if cfg else {}
                st = m.get('state_inventory', {}).get('model')
                rows.append((got == want, m.get('stage'), m.get('progress', {}).get('step'), bool(st and (Path(dp)/st).is_file()), dp))
            except Exception as e:
                rows.append((False, 'ERR '+str(e)[:40], None, False, dp))
    return rows

def phase1_checkpoint(preset):
    if preset in PHASE1_OVERRIDE:
        return Path(PHASE1_OVERRIDE[preset]), {'stage': 'override'}
    for kw in (dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=TOKENIZER_SHA256),
               dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=None),
               dict(stages=('legal_continued','legal_only','wikitext_only','legal','pretrain'), tokenizer_sha256=None)):
        try:
            return resolve_snapshot_checkpoint(COURSE_ROOT/'runs', PRESETS[preset], **kw)
        except FileNotFoundError:
            pass
    rows = _scan_snapshots(preset)
    print(f'--- No usable Phase 1 snapshot for {preset}. {len(rows)} snapshot.json files found under runs/ ---')
    for r in sorted(rows, key=lambda r: (not r[0], str(r[1])))[:25]:
        print('config_match=%s stage=%s step=%s payload_present=%s  %s' % r)
    raise FileNotFoundError(f'{preset}: no snapshot with matching config + payload; set PHASE1_OVERRIDE[{preset!r}] to its state.pt')


def build(C):
    ck, man = phase1_checkpoint(C['preset'])
    m = R.build_model(PRESETS[C['preset']], ck, len(LABELS), 'full' if C['method']=='kd' else C['method'],
                      head_kw=C['head_kw'], adapter_kw=C['adapter_kw'] if C['method'] not in ('full','kd','partial') else {},
                      prefix_len=C['prefix_len'], layer_keep=C['layer_keep'])
    return m, ck



In [ ]:
# Stage 1 · extract frozen sentence vectors (re-tokenised with SENT_CAP)
import numpy as np
for s in DATA.values(): R.tokenize_docs(s, TOK, SENT_CAP)
VEC_DIR.mkdir(parents=True, exist_ok=True)
amp = torch.bfloat16 if (device.type=='cuda' and torch.cuda.is_bf16_supported()) else torch.float16

@torch.no_grad()
def encode(backbone, toks, echo, bs=96):
    out = [None]*len(toks); order = sorted(range(len(toks)), key=lambda i: len(toks[i]))
    for a in range(0, len(order), bs):
        idx = order[a:a+bs]
        seqs = [toks[i]*2 if echo else toks[i] for i in idx]
        T = max(map(len, seqs)); ids = torch.full((len(idx), T), R.PAD_ID, dtype=torch.long)
        for r, q in enumerate(seqs): ids[r, :len(q)] = torch.tensor(q)
        with torch.autocast(device_type=device.type, dtype=amp, enabled=device.type=='cuda'):
            h = R.backbone_hidden(backbone, ids.to(device)).float()
        for r, i in enumerate(idx):
            L = len(toks[i]); end = len(seqs[r]); start = L if echo else 0
            out[i] = torch.cat([h[r, start:end].mean(0), h[r, end-1]]).cpu().numpy().astype(np.float16)
    return out

for preset in STAGE1_PRESETS:
    todo = [p for p in POOLINGS if not (VEC_DIR/f'{preset}_{p}.pt').is_file()]
    if not todo: print(preset, 'cached'); continue
    ck, _ = phase1_checkpoint(preset)
    bb = R.restore_backbone(R.ModelConfig(**PRESETS[preset]).validate(), ck).to(device).eval()
    for pool in todo:
        vec = {}
        for split in ('train','dev','test'):
            for d in DATA[split]:
                vec[d['id']] = encode(bb, d['tok'], pool == 'echo')
        torch.save({k: np.stack(v) for k, v in vec.items()}, VEC_DIR/f'{preset}_{pool}.pt')
        print('saved', preset, pool, len(vec), 'documents')
    del bb; torch.cuda.empty_cache()

In [ ]:
# Stage 2 · grouped k-fold CV of document models on the cached vectors
import doclib as DL, pandas as pd
V = torch.load(VEC_DIR/f"{STAGE2['preset']}_{STAGE2['pooling']}.pt", weights_only=False)
def mk(d): return DL.make_doc(V[d['id']].astype(np.float32), [L2I[l] for l in d['labels']] if d['labels'] else None, d['domain']=='CL')
lab_docs = [mk(d) for d in DATA['train']+DATA['dev']]
mu = np.concatenate([d['x'].numpy() for d in lab_docs]).mean(0); sd = np.concatenate([d['x'].numpy() for d in lab_docs]).std(0)+1e-6
for d in lab_docs: d['x'] = (d['x']-torch.tensor(mu))/torch.tensor(sd)
M = lambda **k: dict(hidden=128, layers=1, dropout=0.3, in_dropout=0.3, **k)
CFGS = {'sentence MLP': dict(model=M(bilstm=False)), 'BiLSTM': dict(model=M()),
        'BiLSTM+shift': dict(model=M(shift=True)), 'BiLSTM+CRF': dict(model=M(crf=True)),
        'BiLSTM+CRF+shift': dict(model=M(crf=True, shift=True))}
rows = []
for name, cfg in CFGS.items():
    cfg['eval_every'] = 5
    curve = DL.cross_validate(lab_docs, cfg, CV_EPOCHS, device, k=CV_K, seeds=CV_SEEDS)[0]
    best = max(curve, key=lambda c: c[1]); cfg['best_epochs'] = best[0]
    rows.append({'model': name, 'best_epoch': best[0], 'CV macro-F1': round(best[1],4), 'CV all-13': round(best[2],4),
                 'last-epoch CV': round(curve[-1][1],4)})
    print(rows[-1])
display(pd.DataFrame(rows))

In [ ]:
# Official-split check (train -> dev), comparable with the earlier per-sentence runs, then the test submission
import copy
CHOSEN = max([r for r in rows if r['model'] in ('BiLSTM','BiLSTM+shift','sentence MLP')], key=lambda r: r['CV macro-F1'])['model']
cfg = CFGS[CHOSEN]; E = cfg['best_epochs']; print('chosen:', CHOSEN, 'epochs', E)
tr = lab_docs[:len(DATA['train'])]; dv = lab_docs[len(DATA['train']):]
m, _ = DL.train_model(tr, cfg, E, device, seed=0)
ydv = np.concatenate([d['y'].numpy() for d in dv]); pdv = np.concatenate(DL.predict(m, dv, device))
print('train->dev macro-F1 (present):', round(DL.macro_f1(ydv, pdv),4), '| all-13:', round(DL.macro_f1(ydv, pdv, True),4))
# test: average probabilities of fold models trained on k-1 folds each (labelled docs = train + dev)
te = [mk(d) for d in DATA['test']]
for d in te: d['x'] = (d['x']-torch.tensor(mu))/torch.tensor(sd)
folds = DL.grouped_folds(len(lab_docs), CV_K, 0); probs = None; n = 0
for f in range(CV_K):
    trn = [lab_docs[i] for j in range(CV_K) if j != f for i in folds[j]]
    for seed in range(3):
        mm, _ = DL.train_model(trn, cfg, E, device, seed=seed)
        p = DL.predict_proba(mm, te, device); n += 1
        probs = p if probs is None else [a+b for a, b in zip(probs, p)]
test_preds = {d['id']: [int(x) for x in pr.argmax(-1)] for d, pr in zip(DATA['test'], probs)}
SUB = RUNS_ROOT/'twostage'/f"{STAGE2['preset']}-{STAGE2['pooling']}-{CHOSEN.replace('+','_').replace(' ','')}"/'submission'
path = R.write_submission(test_preds, DATA['test'], LABELS, SUB)
chk = json.loads(path.read_text()); assert set(chk) == {d['id'] for d in DATA['test']}
print('wrote', SUB/'submission.zip', '| models averaged:', n)